In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd drive/Othercomputers/My\ MacBook\ Pro/GPU-Computing/

/content/drive/Othercomputers/My MacBook Pro/GPU-Computing


In [3]:
!pwd

/content/drive/Othercomputers/My MacBook Pro/GPU-Computing


In [4]:
!which nvcc

/usr/local/cuda/bin/nvcc


In [5]:
!g++ -std=c++17 ./serial_image_blur.cpp -o serial-blur && ./serial-blur

./serial_image_blur.cpp:14:10: fatal error: stb_image.h: No such file or directory
   14 | #include "stb_image.h"
      |          ^~~~~~~~~~~~~
compilation terminated.


In [6]:
!nvcc -arch=sm_75 ./parallel_matrix_mul_tiling_opt.cu -o  mm_tiling_opt

In [7]:
!pwd

/content/drive/Othercomputers/My MacBook Pro/GPU-Computing


In [8]:
!ls


apple.jpg				    README.md
conv					    RGB_to_gray_parallel.cu
convoluted-image.png			    RGB_to_gray_serial.cpp
main					    run.ipynb
matrix_multi_tiling_opt.prof.ncu-rep	    serial-blur
mm_tiling_opt				    serial_blur.png
parallel_blur.png			    serial_gray.png
parallel_gray.png			    serial_image_blur.cpp
parallel_image_blur.cu			    stb
parallel_matrix_mul.cu			    tiled-conv
parallel_matrix_mul_tiling_benchmarking.cu  tiled-convoluted-image.png
parallel_matrix_mul_tiling.cu		    timeline.prof
parallel_matrix_mul_tiling_opt.cu	    vec_profile
parallel-patterns			    vector_addition.cu
profiling.cu


In [9]:
!nvprof -f -o timeline.prof ./vec_profile

======== Error: Application returned non-zero code -1


In [10]:
!nvprof -m all ./vec_profile

======== Warning: Skipping profiling on device 0 since profiling is not supported on devices with compute capability 7.5 and higher.
                  Use NVIDIA Nsight Compute for GPU profiling and NVIDIA Nsight Systems for GPU tracing and CPU sampling.
                  Refer https://developer.nvidia.com/tools-overview for more details.

======== Error: Application returned non-zero code -1


## NVPROF is deprecated

======== Warning: Skipping profiling on device 0 since profiling is not supported on devices with compute capability 7.5 and higher.
                  Use NVIDIA Nsight Compute for GPU profiling and NVIDIA Nsight Systems for GPU tracing and CPU sampling.
                  Refer https://developer.nvidia.com/tools-overview for more details.

We might need NVIDIA Nsight (new profiling) for new devices

In [11]:
!ncu --set full ./mm_tiling_opt

elapsed time CPU 6065.62
==PROF== Connected to process 2202 (/content/drive/Othercomputers/My MacBook Pro/GPU-Computing/mm_tiling_opt)
==PROF== Profiling "mat_mul_tile_kernel_opt" - 0: 0%....50%....100% - 31 passes
elapsed time GPU 7778.84
Max_Diff 9.15527e-05
==PROF== Disconnected from process 2202
[2202] mm_tiling_opt@127.0.0.1
  mat_mul_tile_kernel_opt(float *, float *, float *, int) (8, 32, 1)x(32, 32, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         4.99
    SM Frequency                    Mhz       584.99
    Elapsed Cycles                cycle    2,380,529
    Memory Throughput                 %        80.51
    DRAM Throughput                   %         4.10
    Duration                         ms         4.07
    L1/TEX Cache Throughput 

In [12]:
!ncu --version


NVIDIA (R) Nsight Compute Command Line Profiler
Copyright (c) 2018-2025 NVIDIA Corporation
Version 2025.1.1.0 (build 35528883) (public-release)


##Parallel Patterns

## Convolution


In [20]:
!nvcc -arch=sm_75  -I ./stb ./parallel-patterns/convolution.cu -o conv

./stb/stb_image.h(4276): warning #550-D: variable "old_limit" was set but never used
     unsigned int cur, limit, old_limit;
                              ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

./stb/stb_image.h(5185): warning #550-D: variable "idata_limit_old" was set but never used
                 stbi__uint32 idata_limit_old = idata_limit;
                              ^

./stb/stb_image.h(6972): warning #550-D: variable "out_size" was set but never used
        int out_size = 0;
            ^

./stb/stb_image.h(6973): warning #550-D: variable "delays_size" was set but never used
        int delays_size = 0;
            ^



In [21]:
!./conv

In [15]:
!ncu --set full ./conv

==PROF== Connected to process 2432 (/content/drive/Othercomputers/My MacBook Pro/GPU-Computing/conv)
==PROF== Profiling "convolution_kernel" - 0: 0%....50%....100% - 31 passes
==PROF== Disconnected from process 2432
[2432] conv@127.0.0.1
  convolution_kernel(unsigned char *, unsigned char *, unsigned int, unsigned int) (32, 32, 1)x(32, 32, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         4.99
    SM Frequency                    Mhz       584.99
    Elapsed Cycles                cycle      440,668
    Memory Throughput                 %        29.06
    DRAM Throughput                   %         5.04
    Duration                         us       753.28
    L1/TEX Cache Throughput           %        31.31
    L2 Cache Throughput               %    

Convolution with tiling

In [34]:
!nvcc -arch=sm_75  -I ./stb ./parallel-patterns/tiled-convolution.cu -o tiled-conv

./stb/stb_image.h(4276): warning #550-D: variable "old_limit" was set but never used
     unsigned int cur, limit, old_limit;
                              ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

./stb/stb_image.h(5185): warning #550-D: variable "idata_limit_old" was set but never used
                 stbi__uint32 idata_limit_old = idata_limit;
                              ^

./stb/stb_image.h(6972): warning #550-D: variable "out_size" was set but never used
        int out_size = 0;
            ^

./stb/stb_image.h(6973): warning #550-D: variable "delays_size" was set but never used
        int delays_size = 0;
            ^



In [37]:
!./tiled-conv

In [38]:
!ls

apple.jpg				    README.md
conv					    RGB_to_gray_parallel.cu
convoluted-image.png			    RGB_to_gray_serial.cpp
main					    run.ipynb
matrix_multi_tiling_opt.prof.ncu-rep	    serial-blur
mm_tiling_opt				    serial_blur.png
parallel_blur.png			    serial_gray.png
parallel_gray.png			    serial_image_blur.cpp
parallel_image_blur.cu			    stb
parallel_matrix_mul.cu			    tiled-conv
parallel_matrix_mul_tiling_benchmarking.cu  tiled-convoluted-image.png
parallel_matrix_mul_tiling.cu		    timeline.prof
parallel_matrix_mul_tiling_opt.cu	    vec_profile
parallel-patterns			    vector_addition.cu
profiling.cu


In [19]:
!df

Filesystem     1K-blocks     Used Available Use% Mounted on
overlay        118108032 45706564  72385084  39% /
tmpfs              65536        0     65536   0% /dev
shm              5958656        0   5958656   0% /dev/shm
/dev/root        2019696  1253912    765784  63% /usr/sbin/docker-init
tmpfs            6643472       64   6643408   1% /var/colab
/dev/sda1       77160852 50847040  26297428  66% /kaggle/input
tmpfs            6643472        0   6643472   0% /proc/acpi
tmpfs            6643472        0   6643472   0% /proc/scsi
tmpfs            6643472        0   6643472   0% /sys/firmware
drive          118108032 49342204  68765828  42% /content/drive


In [39]:
!ncu --set full ./tiled-conv

==PROF== Connected to process 14535 (/content/drive/Othercomputers/My MacBook Pro/GPU-Computing/tiled-conv)
==PROF== Profiling "tiled_convolution_kernel" - 0: 0%....50%....100% - 31 passes
==PROF== Disconnected from process 14535
[14535] tiled-conv@127.0.0.1
  tiled_convolution_kernel(unsigned char *, unsigned char *, unsigned int, unsigned int) (63, 63, 1)x(20, 20, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         5.00
    SM Frequency                    Mhz       584.96
    Elapsed Cycles                cycle      225,002
    Memory Throughput                 %        84.22
    DRAM Throughput                   %        10.95
    Duration                         us       384.64
    L1/TEX Cache Throughput           %        91.78
    L2 Cache Thr